In [0]:
# Configuration

from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from datetime import datetime

RAW_PATH = "/Volumes/education_analytics/default/education_analytics_platform/enrollment_data_raw.csv"
DATABASE_NAME = "education_db"
BRONZE_TABLE  = f"{DATABASE_NAME}.bronze_enrollment"
RUN_ID        = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BRONZE LAYER CONFIGURATION")
print("=" * 50)
print(f"Source file  : {RAW_PATH}")
print(f"Target table : {BRONZE_TABLE}")
print(f"Run ID       : {RUN_ID}")


In [0]:
# Create Database

# Create education_db if it doesn't already exist
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")
spark.sql(f"USE {DATABASE_NAME}")
print(f"Database '{DATABASE_NAME}' is ready")


In [0]:
# Load Raw CSV (All Columns as Strings)

# inferSchema=False is intentional — Bronze preserves the raw data exactly
# as it came from the source, including all dirty values, wrong types, and
# inconsistencies. This is the correct Medallion Architecture pattern.
df_raw = spark.read.csv(
    RAW_PATH,
    header=True,
    inferSchema=False,   # Keep ALL values as strings — no type assumptions
    multiLine=True,
    escape='"'
)

print(f"Raw CSV loaded:")
print(f"  Rows    : {df_raw.count():,}")
print(f"  Columns : {len(df_raw.columns)}")
print(f"\nSchema (all StringType — intentional):")
df_raw.printSchema()

In [0]:
# Add Bronze Metadata Columns

# Metadata columns track WHERE and WHEN each row was ingested.
# These are essential for data lineage — if something goes wrong,
# you can always trace a record back to its source file and run.
df_bronze = df_raw \
    .withColumn("_ingestion_timestamp", F.current_timestamp()) \
    .withColumn("_source_file",         F.lit(RAW_PATH)) \
    .withColumn("_pipeline_run_id",     F.lit(RUN_ID)) \
    .withColumn("_layer",               F.lit("bronze"))

print("Metadata columns added:")
print("  _ingestion_timestamp — when this row was loaded")
print("  _source_file         — which file it came from")
print("  _pipeline_run_id     — which pipeline run ingested it")
print("  _layer               — always 'bronze'")


In [0]:
# Write to Bronze Delta Table

# mode("overwrite") replaces previous Bronze data on each pipeline run.
# This is correct — Bronze always reflects the latest uploaded raw file.
# overwriteSchema=True allows schema changes between runs.
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(BRONZE_TABLE)

bronze_count = df_bronze.count()
print("BRONZE INGESTION COMPLETE")
print("=" * 50)
print(f"Table        : {BRONZE_TABLE}")
print(f"Rows written : {bronze_count:,}")
print(f"Run ID       : {RUN_ID}")


In [0]:
# Verify: Read Back and Preview

df_verify = spark.table(BRONZE_TABLE)

print(f"Bronze Delta table row count: {df_verify.count():,}")
print(f"\nFirst 5 rows:")
df_verify.select(
    "record_id", "school_name", "region", "academic_year",
    "grade", "subject", "total_enrollment",
    "avg_performance_score", "dropout_rate",
    "_ingestion_timestamp", "_layer"
).display(5, truncate=False)

In [0]:
# Quality Flag Distribution in Bronze

# Show all data quality issues present in Bronze (intentionally preserved)
print("Data quality flag distribution (all issues visible in Bronze):")
df_verify.groupBy("data_quality_flag") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)


In [0]:
# Bronze Delta Table History

# Delta Lake tracks every write operation — this shows the full history
print("Bronze Delta Table History:")
spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}").display(5, truncate=False)


